Demanda / estacionalidad

1. Viajes por mes (2024)
-- Tablas usadas:
-- analytics_gold.fact_trips
-- analytics_gold.dim_date

In [ ]:
select
    d.year,
    d.month,
    count(*) as total_trips
from analytics_gold.fact_trips f
join analytics_gold.dim_date d
    on f.pickup_date_key = d.date_key
where d.year = 2021
group by d.year, d.month
order by d.month;

La demanda se concentra casi por completo en enero y febrero de 2021, con 1,433,830 y 1,423,063 viajes respectivamente.

Marzo presenta solo 3 registros, por lo que no representa un mes comparable y probablemente corresponde a datos residuales o incompletos.

En términos generales, el volumen mensual se mantiene muy estable entre enero y febrero dentro del periodo cargado en la capa gold.

2. Viajes por service_type y mes
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date
-- analitics_gold.dim_service_type

In [ ]:
select
    d.year,
    d.month,
    st.service_type,
    count(*) as total_trips
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
join analitics_gold.dim_service_type st
    on f.service_type_key = st.service_type_key
where d.year = 2021
group by d.year, d.month, st.service_type
order by d.month, st.service_type;

Yellow domina claramente la demanda en los meses analizados, con 1,357,449 viajes en enero y 1,358,604 en febrero, muy por encima de green.
Green representa una fracción mucho menor del total, con 76,381 viajes en enero y 64,459 en febrero, además de mostrar una caída entre ambos meses.
Marzo no es comparable porque solo registra 3 viajes en total, por lo que el análisis útil se concentra en enero y febrero de 2021.

3. Top 10 zonas de pickup (total 2021)
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date
-- analitics_gold.dim_zone

In [ ]:
select
    z.zone as pickup_zone,
    z.borough,
    count(*) as total_trips
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
join analitics_gold.dim_zone z
    on f.pu_location_key = z.zone_key
where d.year = 2021
group by z.zone, z.borough
order by total_trips desc
limit 10;

Las 10 zonas con más pickups pertenecen todas a Manhattan, lo que muestra una concentración espacial muy fuerte de la demanda en este borough.
Las zonas líderes son Upper East Side North y Upper East Side South, con 149,066 y 145,367 viajes respectivamente, seguidas por áreas como Lenox Hill West y Upper West Side South.
Esto sugiere que los principales puntos de origen del periodo analizado se concentran en zonas residenciales y comerciales de alta actividad dentro de Manhattan.

4. Top 10 zonas de dropoff (total 2021)
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date
-- analitics_gold.dim_zone

In [ ]:
select
    z.zone as dropoff_zone,
    z.borough,
    count(*) as total_trips
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
join analitics_gold.dim_zone z
    on f.do_location_key = z.zone_key
where d.year = 2021
group by z.zone, z.borough
order by total_trips desc
limit 10;

Las 10 zonas con más dropoffs también pertenecen completamente a Manhattan, lo que confirma que este borough concentra no solo los principales orígenes, sino también los destinos más frecuentes de los viajes.
La zona con mayor número de llegadas es Upper East Side North con 149,103 viajes, seguida por Upper East Side South con 125,071 y Lenox Hill West con 89,311.
Al comparar este ranking con el de pickups, se observa que varias zonas aparecen en ambos listados, lo que sugiere una movilidad intensa dentro de áreas muy activas de Manhattan.

5. Top 5 boroughs por mes (pickup)
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date
-- analitics_gold.dim_zone

In [ ]:
with borough_month as (
    select
        d.year,
        d.month,
        z.borough,
        count(*) as total_trips
    from analitics_gold.fact_trips f
    join analitics_gold.dim_date d
        on f.pickup_date_key = d.date_key
    join analitics_gold.dim_zone z
        on f.pu_location_key = z.zone_key
    where d.year = 2021
    group by d.year, d.month, z.borough
),
ranked as (
    select
        year,
        month,
        borough,
        total_trips,
        row_number() over (
            partition by year, month
            order by total_trips desc
        ) as ranking
    from borough_month
)
select
    year,
    month,
    borough,
    total_trips,
    ranking
from ranked
where ranking <= 5
order by month, ranking;

Manhattan lidera claramente el número de pickups en enero y febrero de 2021, con 1,254,653 y 1,267,762 viajes respectivamente, manteniéndose en el primer lugar en ambos meses.
Queens ocupa el segundo puesto en enero y febrero, seguido por Brooklyn y Bronx, lo que muestra una jerarquía espacial bastante estable en la demanda.
La categoría Unknown aparece en el quinto lugar, lo que sugiere que existe una pequeña proporción de registros sin borough identificado; marzo no es representativo porque solo contiene 3 viajes en total.

6. Horas pico (top 5 horas) para cada día de semana
-- Tablas usadas:
-- analitics_gold.fact_trips

In [ ]:
with hourly_demand as (
    select
        extract(dow from pickup_at) as day_of_week_num,
        trim(to_char(pickup_at, 'Day')) as day_name,
        extract(hour from pickup_at) as pickup_hour,
        count(*) as total_trips
    from analitics_gold.fact_trips
    where extract(year from pickup_at) = 2021
    group by 1, 2, 3
),
ranked as (
    select
        day_of_week_num,
        day_name,
        pickup_hour,
        total_trips,
        row_number() over (
            partition by day_of_week_num
            order by total_trips desc
        ) as ranking
    from hourly_demand
)
select
    day_of_week_num,
    day_name,
    pickup_hour,
    total_trips,
    ranking
from ranked
where ranking <= 5
order by day_of_week_num, ranking;

Las horas pico se concentran de forma consistente en la tarde, especialmente entre las 14:00 y 18:00, lo que indica que la mayor demanda ocurre en franjas de alta movilidad diaria.
En los días observados, Friday a las 15:00 presenta uno de los mayores picos con 38,147 viajes, seguido por valores altos también en Wednesday y Thursday en ese mismo rango horario.
En general, el patrón horario es bastante estable entre días de semana y sábado, con una clara concentración de viajes en horas de la tarde más que en la mañana o la noche.

7. Distribución de viajes por día de semana (ranking)
-- Tablas usadas:
-- analitics_gold.fact_trips

In [ ]:
select
    extract(dow from pickup_at) as day_of_week_num,
    trim(to_char(pickup_at, 'Day')) as day_name,
    count(*) as total_trips,
    rank() over (
        order by count(*) desc
    ) as ranking
from analitics_gold.fact_trips
where extract(year from pickup_at) = 2021
group by 1, 2
order by ranking;

Friday es el día con mayor volumen de viajes, con 497,203, seguido por Wednesday con 456,398 y Thursday con 450,329, lo que muestra que la mayor demanda se concentra hacia el final de la semana laboral.
Saturday también mantiene una actividad alta con 406,581 viajes, mientras que Sunday registra el menor volumen con 301,780.
En conjunto, el patrón semanal sugiere que la movilidad es más intensa en días laborales medios y finales, y disminuye al inicio de la semana y especialmente el domingo.

Ingresos / tarifas / propinas

8. Ingreso total (total_amount) por mes
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date

In [ ]:
select
    d.year,
    d.month,
    round(sum(f.total_amount)::numeric, 2) as total_revenue
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
where d.year = 2021
group by d.year, d.month
order by d.month;

El ingreso total se concentra casi por completo en enero y febrero de 2021, con 25,690,287.85 y 25,558,987.29 respectivamente, lo que muestra un nivel de revenue muy similar entre ambos meses.
Marzo registra apenas 63.19, por lo que no representa un periodo comparable y probablemente corresponde a datos residuales o incompletos.
En términos generales, el comportamiento del ingreso total es estable entre enero y febrero dentro del periodo efectivamente cargado en la capa gold.

9. Ingreso total por service_type y mes
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date
-- analitics_gold.dim_service_type

In [ ]:
select
    d.year,
    d.month,
    st.service_type,
    round(sum(f.total_amount)::numeric, 2) as total_revenue
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
join analitics_gold.dim_service_type st
    on f.service_type_key = st.service_type_key
where d.year = 2021
group by d.year, d.month, st.service_type
order by d.month, st.service_type;

Yellow aporta la mayor parte del ingreso total en ambos meses principales del análisis, con 23,885,069.97 en enero y 24,044,448.58 en febrero, muy por encima de green.
Green genera 1,805,217.88 en enero y 1,514,538.71 en febrero, por lo que su contribución al revenue total es considerablemente menor y además disminuye de un mes a otro.
Marzo no es comparable porque solo contiene ingresos residuales, así que el análisis útil se concentra en enero y febrero de 2021.

10. tip % promedio por mes
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date

In [ ]:
select
    d.year,
    d.month,
    round(avg(f.tip_amount / nullif(f.fare_amount, 0))::numeric * 100, 2) as avg_tip_pct
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
where d.year = 2021
group by d.year, d.month
order by d.month;

El porcentaje promedio de propina se mantiene bastante estable entre enero y febrero de 2021, con 19.62% y 19.42% respectivamente, lo que sugiere un comportamiento muy similar de los usuarios en ambos meses.
Marzo presenta un valor de 16.91%, pero no es un mes comparable debido a la cantidad mínima de registros disponibles.
En términos generales, durante el periodo útil del análisis la propina promedio representa cerca de una quinta parte del fare_amount.

11. tip % por borough y mes
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date
-- analitics_gold.dim_zone

In [ ]:
select
    d.year,
    d.month,
    z.borough,
    round(avg(f.tip_amount / nullif(f.fare_amount, 0))::numeric * 100, 2) as avg_tip_pct
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
join analitics_gold.dim_zone z
    on f.pu_location_key = z.zone_key
where d.year = 2021
group by d.year, d.month, z.borough
order by d.month, avg_tip_pct desc;

El borough con el tip % promedio más alto en enero y febrero es EWR, con 79.00% y 131.22%, pero estos valores deben interpretarse con cautela porque probablemente corresponden a un volumen muy bajo de viajes y pueden estar distorsionados por outliers.
Entre los boroughs con actividad más consistente, Manhattan muestra el comportamiento más alto y estable, con 20.77% en enero y 20.78% en febrero, seguido por Unknown y luego Queens.
En contraste, Brooklyn y Bronx presentan porcentajes de propina más bajos, lo que sugiere diferencias espaciales en el comportamiento relativo de las propinas dentro del periodo analizado.

12. Top 10 zonas por ingreso total (pickup)
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date
-- analitics_gold.dim_zone

In [ ]:
select
    z.zone as pickup_zone,
    z.borough,
    round(sum(f.total_amount)::numeric, 2) as total_revenue
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
join analitics_gold.dim_zone z
    on f.pu_location_key = z.zone_key
where d.year = 2021
group by z.zone, z.borough
order by total_revenue desc
limit 10;

La zona con mayor ingreso total es JFK Airport en Queens, con 2,941,121.49, superando incluso a las zonas de Manhattan con mayor volumen de viajes.
El resto del top 10 está dominado por zonas de Manhattan como Upper East Side North, Upper East Side South y Lenox Hill East, lo que confirma una alta concentración económica en este borough.
Esto sugiere que no siempre las zonas con más viajes son las que más facturan, ya que trayectos asociados a aeropuertos pueden generar ingresos más altos por viaje.

13. Top 10 zonas por tip % (pickup) con mínimo N viajes
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date
-- analitics_gold.dim_zone
-- N definido: 1000 viajes mínimos

In [ ]:
select
    z.zone as pickup_zone,
    z.borough,
    count(*) as total_trips,
    round(avg(f.tip_amount / nullif(f.fare_amount, 0))::numeric * 100, 2) as avg_tip_pct
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
join analitics_gold.dim_zone z
    on f.pu_location_key = z.zone_key
where d.year = 2021
group by z.zone, z.borough
having count(*) >= 1000
order by avg_tip_pct desc
limit 10;

Al exigir un mínimo de 1000 viajes, las zonas con mayor tip % promedio se concentran completamente en Manhattan, lideradas por Midtown North con 26.05% y West Chelsea/Hudson Yards con 25.18%.
También destacan zonas como UN/Turtle Bay South, Yorkville East y Midtown East, todas con porcentajes superiores al 23%, lo que sugiere un patrón consistente de propinas altas en áreas centrales y de alta actividad económica.
Esto indica que, entre las zonas con volumen suficiente, Manhattan no solo concentra muchos viajes, sino también mejores niveles relativos de propina.

14. Comparación cash vs card: viajes, ingreso total, tip %
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_payment_type

In [ ]:
select
    pt.payment_type,
    count(*) as total_trips,
    round(sum(f.total_amount)::numeric, 2) as total_revenue,
    round(avg(f.tip_amount / nullif(f.fare_amount, 0))::numeric * 100, 2) as avg_tip_pct
from analitics_gold.fact_trips f
join analitics_gold.dim_payment_type pt
    on f.payment_type_key = pt.payment_type_key
where pt.payment_type in ('cash', 'card')
group by pt.payment_type
order by total_trips desc;

El pago con card domina claramente sobre cash en las tres métricas analizadas: registra 1,884,339 viajes frente a 630,248, genera 32,753,483.48 de ingreso total frente a 9,138,196.30, y además presenta un tip % promedio de 28.09%.
En contraste, cash muestra un tip % de 0.00%, lo que es consistente con datasets de taxis donde muchas propinas quedan registradas principalmente en pagos con tarjeta.
En conjunto, esto sugiere que card no solo es el medio de pago más usado, sino también el más relevante para el revenue y para la captura de propinas.

15. Duración promedio (min) por mes
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date

In [ ]:
select
    d.year,
    d.month,
    round(avg(f.trip_duration_min)::numeric, 2) as avg_trip_duration_min
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
where d.year = 2021
group by d.year, d.month
order by d.month;

La duración promedio de los viajes aumenta ligeramente de 14.40 minutos en enero a 15.27 minutos en febrero, lo que sugiere trayectos un poco más largos o condiciones de movilidad menos ágiles en febrero.
Marzo muestra una duración promedio de 77.91 minutos, pero ese valor no es representativo debido al número mínimo de registros disponibles en ese mes.
Por lo tanto, el análisis útil indica que los viajes en enero y febrero de 2021 duran en promedio entre 14 y 15 minutos.

16. Distancia promedio por mes
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_date

In [ ]:
select
    d.year,
    d.month,
    round(avg(f.trip_distance)::numeric, 2) as avg_trip_distance
from analitics_gold.fact_trips f
join analitics_gold.dim_date d
    on f.pickup_date_key = d.date_key
where d.year = 2021
group by d.year, d.month
order by d.month;

La distancia promedio de los viajes disminuye de 6.57 millas en enero a 4.91 millas en febrero, lo que sugiere que en febrero los trayectos fueron, en promedio, más cortos.
Marzo presenta una distancia promedio de 9.97 millas, pero ese valor no es representativo debido al número mínimo de registros disponibles en ese mes.
En términos generales, durante el periodo útil del análisis los viajes se mantienen dentro de un rango moderado, con una reducción clara en la longitud promedio entre enero y febrero.

17. Velocidad promedio (mph) por borough y franja horaria
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_zone

In [ ]:
select
    z.borough,
    case
        when extract(hour from f.pickup_at) between 0 and 5 then 'madrugada'
        when extract(hour from f.pickup_at) between 6 and 11 then 'manana'
        when extract(hour from f.pickup_at) between 12 and 17 then 'tarde'
        else 'noche'
    end as time_band,
    round(avg(
        f.trip_distance / nullif(f.trip_duration_min / 60.0, 0)
    )::numeric, 2) as avg_mph
from analitics_gold.fact_trips f
join analitics_gold.dim_zone z
    on f.pu_location_key = z.zone_key
where extract(year from f.pickup_at) = 2021
  and f.trip_duration_min > 0
group by z.borough, time_band
order by z.borough, time_band;

La velocidad promedio más baja y consistente se observa en Manhattan, donde varía entre 12.95 mph en la tarde y 20.32 mph en la madrugada, lo que sugiere mayor congestión y menor eficiencia operativa en este borough.
En contraste, boroughs como Bronx, Queens y especialmente EWR muestran velocidades mucho más altas, aunque estos valores deben interpretarse con cautela porque probablemente están influenciados por outliers o por un menor volumen de viajes.
En general, la franja de tarde tiende a registrar velocidades menores que madrugada o mañana, lo que es consistente con una mayor congestión durante horas de mayor actividad.

18. Percentiles p50 y p90 de duración por borough
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_zone

In [ ]:
select
    z.borough,
    round(
        percentile_cont(0.50) within group (order by f.trip_duration_min)::numeric,
        2
    ) as p50_duration_min,
    round(
        percentile_cont(0.90) within group (order by f.trip_duration_min)::numeric,
        2
    ) as p90_duration_min
from analitics_gold.fact_trips f
join analitics_gold.dim_zone z
    on f.pu_location_key = z.zone_key
where extract(year from f.pickup_at) = 2021
  and f.trip_duration_min is not null
group by z.borough
order by p90_duration_min desc;

Staten Island presenta las duraciones más altas, con un p50 de 46.98 min y un p90 de 70.50 min, lo que indica que incluso sus viajes típicos son considerablemente más largos que en otros boroughs.
Brooklyn, Queens y Bronx muestran valores intermedios, con percentiles 90 entre aproximadamente 40 y 43 minutos, mientras que Manhattan tiene trayectos claramente más cortos, con p50 de 8.98 min y p90 de 20.43 min.
EWR registra tiempos extremadamente bajos, por lo que sus valores deben interpretarse con cautela, ya que probablemente están influenciados por outliers o por pocos registros.

19. Top 10 zonas (pickup) por p90 de duración
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_zone

In [ ]:
select
    z.zone as pickup_zone,
    z.borough,
    round(
        percentile_cont(0.90) within group (order by f.trip_duration_min)::numeric,
        2
    ) as p90_duration_min
from analitics_gold.fact_trips f
join analitics_gold.dim_zone z
    on f.pu_location_key = z.zone_key
where extract(year from f.pickup_at) = 2021
  and f.trip_duration_min is not null
group by z.zone, z.borough
order by p90_duration_min desc
limit 10;

Las zonas con mayor p90 de duración se concentran principalmente en Staten Island, lideradas por Arden Heights con 82.55 min, seguida por New Dorp/Midland Beach con 76.98 min y Bloomfield/Emerson Hill con 76.66 min.
También aparecen algunas zonas de Queens como Bellerose y Broad Channel, pero el patrón dominante muestra que una parte importante de los viajes más largos se origina en Staten Island.
Esto sugiere que estas zonas están asociadas a trayectos extensos o a condiciones donde los viajes tienden a durar mucho más en la cola alta de la distribución.

20. Top 10 rutas borough → borough por número de viajes
-- Tablas usadas:
-- analitics_gold.fact_trips
-- analitics_gold.dim_zone

In [ ]:
select
    zpu.borough as pickup_borough,
    zdo.borough as dropoff_borough,
    count(*) as total_trips
from analitics_gold.fact_trips f
join analitics_gold.dim_zone zpu
    on f.pu_location_key = zpu.zone_key
join analitics_gold.dim_zone zdo
    on f.do_location_key = zdo.zone_key
where extract(year from f.pickup_at) = 2021
group by zpu.borough, zdo.borough
order by total_trips desc
limit 10;

La ruta más frecuente es claramente Manhattan → Manhattan, con 2,366,652 viajes, lo que muestra que la movilidad se concentra fuertemente dentro del mismo borough.
También destacan rutas internas como Queens → Queens y Brooklyn → Brooklyn, aunque con volúmenes muy inferiores, y luego aparecen conexiones interborough importantes como Manhattan → Brooklyn, Manhattan → Queens y Queens → Manhattan.
En conjunto, el patrón sugiere que la mayor parte de los desplazamientos ocurre dentro de cada borough, especialmente en Manhattan, mientras que las rutas entre boroughs tienen un peso secundario pero aún relevante.